In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

In [2]:
df = pd.read_excel("/Users/rupamhaldar/Desktop/CChunP/data/telco_cleaned.xlsx")

In [3]:
df.head()

,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,Senior Citizen,Partner,...,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV
0,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,No,No,...,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239
1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,No,No,...,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701
2,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,No,No,...,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372
3,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,No,Yes,...,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003
4,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,No,No,...,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340


In [7]:
# Drop irrelevant columns

# Drop geographic columns 
drop_cols = ['Country','State','City','Zip Code','Lat Long','Latitude','Longitude']
df.drop(columns=drop_cols, inplace = True, errors='ignore')

# Drop ChurnValue and ChurnScore as they are derived from the target variable (ChurnLabel),
# Keeping them would cause data leakage and give the model the answer directly.

df.drop(columns = ['Churn Value', 'Churn Score'], inplace = True, errors='ignore')
             
# Check the shape
print(df.shape)

(7043, 21)


In [8]:
# Handle missing or inconsistent values

# 'Total Charges' and 'Monthly Charges'may have blank strings → convert to numeric; invalid parsing becomes NaN
cols_to_numeric = ['Total Charges', 'Monthly Charges']
df[cols_to_numeric] = df[cols_to_numeric].apply(pd.to_numeric, errors='coerce')

df.isnull().sum()


Gender                0
Senior Citizen        0
Partner               0
Dependents            0
Tenure Months         0
Phone Service         0
Multiple Lines        0
Internet Service      0
Online Security       0
Online Backup         0
Device Protection     0
Tech Support          0
Streaming TV          0
Streaming Movies      0
Contract              0
Paperless Billing     0
Payment Method        0
Monthly Charges       0
Total Charges        11
Churn Label           0
CLTV                  0
dtype: int64

In [9]:
# Drop rows with missing Total Charges
df = df.dropna()


In [10]:
df.isnull().sum()

Gender               0
Senior Citizen       0
Partner              0
Dependents           0
Tenure Months        0
Phone Service        0
Multiple Lines       0
Internet Service     0
Online Security      0
Online Backup        0
Device Protection    0
Tech Support         0
Streaming TV         0
Streaming Movies     0
Contract             0
Paperless Billing    0
Payment Method       0
Monthly Charges      0
Total Charges        0
Churn Label          0
CLTV                 0
dtype: int64

In [11]:
df.head(5)

,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,...,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,CLTV
0,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,3239
1,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,2701
2,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,...,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,Yes,5372
3,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,...,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,5003
4,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,...,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,Yes,5340


In [14]:
# Encode binary categorical variables

df = df.copy()

binary_cols = ['Gender','Senior Citizen','Partner','Dependents','Phone Service','Paperless Billing','Churn Label',
               'Multiple Lines','Online Security','Online Backup','Device Protection','Tech Support','Streaming TV','Streaming Movies']


for col in binary_cols:
    df[col] = df[col].replace({
        'Yes':1, 'No':0,
        'Male':1, 'Female':0,
        'No internet service':0,
        'No phone service':0
    })
    
# Quick check
df[binary_cols].head()


,Gender,Senior Citizen,Partner,Dependents,Phone Service,Paperless Billing,Churn Label,Multiple Lines,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies
0,1,0,0,0,1,1,1,0,1,1,0,0,0,0
1,0,0,0,1,1,1,1,0,0,0,0,0,0,0
2,0,0,0,1,1,1,1,1,0,0,1,0,1,1
3,0,0,1,1,1,1,1,1,0,0,1,1,1,1
4,1,0,0,1,1,1,1,1,0,1,1,0,1,1


In [15]:
# Step 4: One-hot encode multi-category columns

multi_cat_cols = ['Internet Service','Contract','Payment Method']

df = pd.get_dummies(df, columns=multi_cat_cols, drop_first=True)

# Verify all columns are numeric
df.info()

<class 'pandas.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 25 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   Gender                                  7032 non-null   object 
 1   Senior Citizen                          7032 non-null   object 
 2   Partner                                 7032 non-null   object 
 3   Dependents                              7032 non-null   object 
 4   Tenure Months                           7032 non-null   int64  
 5   Phone Service                           7032 non-null   object 
 6   Multiple Lines                          7032 non-null   object 
 7   Online Security                         7032 non-null   object 
 8   Online Backup                           7032 non-null   object 
 9   Device Protection                       7032 non-null   object 
 10  Tech Support                            7032 non-null   object 
 11  Streami

In [18]:
# Convert boolean columns to int
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

# Check
df.head()

,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Online Security,Online Backup,Device Protection,...,Total Charges,Churn Label,CLTV,Internet Service_Fiber optic,Internet Service_No,Contract_One year,Contract_Two year,Payment Method_Credit card (automatic),Payment Method_Electronic check,Payment Method_Mailed check
0,1,0,0,0,2,1,0,1,1,0,...,108.15,1,3239,0,0,0,0,0,0,1
1,0,0,0,1,2,1,0,0,0,0,...,151.65,1,2701,1,0,0,0,0,1,0
2,0,0,0,1,8,1,1,0,0,1,...,820.50,1,5372,1,0,0,0,0,1,0
3,0,0,1,1,28,1,1,0,0,1,...,3046.05,1,5003,1,0,0,0,0,1,0
4,1,0,0,1,49,1,1,0,1,1,...,5036.30,1,5340,1,0,0,0,0,0,0


In [21]:
# Step 5: Scale numeric features


scaler = StandardScaler()
scale_cols = ['Tenure Months','Monthly Charges','Total Charges','CLTV']

df[scale_cols] = scaler.fit_transform(df[scale_cols])

# Quick check
df[scale_cols].head()

,Tenure Months,Monthly Charges,Total Charges,CLTV
0,-1.239504,-0.363923,-0.959649,-0.983181
1,-1.239504,0.196178,-0.940457,-1.438215
2,-0.995040,1.158489,-0.645369,0.820883
3,-0.180161,1.329677,0.336516,0.508788
4,0.675462,1.293113,1.214589,0.793818


In [22]:
df.head()

,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Online Security,Online Backup,Device Protection,...,Total Charges,Churn Label,CLTV,Internet Service_Fiber optic,Internet Service_No,Contract_One year,Contract_Two year,Payment Method_Credit card (automatic),Payment Method_Electronic check,Payment Method_Mailed check
0,1,0,0,0,-1.239504,1,0,1,1,0,...,-0.959649,1,-0.983181,0,0,0,0,0,0,1
1,0,0,0,1,-1.239504,1,0,0,0,0,...,-0.940457,1,-1.438215,1,0,0,0,0,1,0
2,0,0,0,1,-0.995040,1,1,0,0,1,...,-0.645369,1,0.820883,1,0,0,0,0,1,0
3,0,0,1,1,-0.180161,1,1,0,0,1,...,0.336516,1,0.508788,1,0,0,0,0,1,0
4,1,0,0,1,0.675462,1,1,0,1,1,...,1.214589,1,0.793818,1,0,0,0,0,0,0


In [23]:
df.info()

<class 'pandas.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 25 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   Gender                                  7032 non-null   object 
 1   Senior Citizen                          7032 non-null   object 
 2   Partner                                 7032 non-null   object 
 3   Dependents                              7032 non-null   object 
 4   Tenure Months                           7032 non-null   float64
 5   Phone Service                           7032 non-null   object 
 6   Multiple Lines                          7032 non-null   object 
 7   Online Security                         7032 non-null   object 
 8   Online Backup                           7032 non-null   object 
 9   Device Protection                       7032 non-null   object 
 10  Tech Support                            7032 non-null   object 
 11  Streami

In [24]:
df.to_excel("/Users/rupamhaldar/Desktop/CChunP/data/telco_preprocessed.xlsx", index=False)